__No masking (except for the target mask)__

In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/main_cumulative_lrg.fits'))
cat['EFFTIME_ELG'] = 8.60 * cat['TSNR2_ELG']
cat['EFFTIME_LRG'] = 12.15 * cat['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Remove "no data" fibers
mask = cat['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# # Apply LRG mask
# mask = cat['lrg_mask']==0
# print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# cat = cat[mask]

# # Remove QSO targets
# mask = cat['DESI_TARGET'] & 2**2 ==0
# print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# cat = cat[mask]

# Require a minimum depth
min_depth = 800.
mask = cat['EFFTIME_LRG']>min_depth
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]

# # Julien's bad fibers list + my list of worst-performing fibers; bad_fibers-everest.ipynb
# # bad_fibers = np.loadtxt('/global/cfs/cdirs/desi/users/rongpu/spectro/everest/misc/bad_fibers_20211117.txt', dtype=int)
# bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211117.txt', dtype=int)
# print(len(bad_fibers))
# mask_bad = np.in1d(cat['FIBER'], bad_fibers)
# print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
# cat = cat[~mask_bad]
# print(len(cat), len(np.unique(cat['TARGETID'])))

FIBERSTATUS 338266 7165 0.020742203218587794
No data 338265 1 2.9562533627382e-06
Min depth 325438 12827 0.9620800260151064


In [4]:
# Redshift quality cut
cat['q'] = cat['ZWARN']==0
cat['q'] &= cat['Z']<1.5
cat['q'] &= cat['DELTACHI2']>15

print(np.sum(~cat['q'])/len(cat['q']))
# cat = cat[cat['q']]
# print(len(cat))

0.014841536636778742


In [5]:
mask_star = (cat['SPECTYPE']=='STAR') | (cat['Z']<0.0003)
cat['SPECTYPE'][mask_star] = 'STAR'
print(np.sum(mask_star)/len(mask_star))

0.015569785950011983


In [6]:
t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'], return_counts=True)
t['frac (%)'] = t['count']/len(cat)*100
t['frac (%)'].format = '%.3f'
t.sort('count')
t

SPECTYPE,count,frac (%)
str6,int64,float64
STAR,5067,1.557
QSO,6891,2.117
GALAXY,313480,96.326


In [7]:
# Fraction of QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask)/len(mask))

0.01607372218364174


In [8]:
# Exclude QSO targets
mask = cat['DESI_TARGET'] & 2**2==0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.3f'
t.sort('count')
t

320207 0.9839262778163582


SPECTYPE,count,frac (%)
str6,int64,float64
QSO,3796,1.185
STAR,4793,1.497
GALAXY,311618,97.318


In [9]:
# Select QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.3f'
t.sort('count')
t

5231 0.01607372218364174


SPECTYPE,count,frac (%)
str6,int64,float64
STAR,274,5.238
GALAXY,1862,35.595
QSO,3095,59.167


In [10]:
# Downweight QSO targets and recalculate the fractions
mask_qso = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask_qso)/len(mask_qso))
cat['weight'] = 1.
cat['weight'][mask_qso] = 0.5

mask = cat['SPECTYPE']=='STAR'
print('Star fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))
mask = cat['SPECTYPE']=='QSO'
print('QSO fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))

0.01607372218364174
Star fraction: 1.53%
QSO fraction: 1.66%


------
# Spectype of masked LRG targets

In [11]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/main_cumulative_lrg.fits'))
cat['EFFTIME_ELG'] = 8.60 * cat['TSNR2_ELG']
cat['EFFTIME_LRG'] = 12.15 * cat['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Remove "no data" fibers
mask = cat['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# # Apply LRG mask
# mask = cat['lrg_mask']!=0
# print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# cat = cat[mask]

# # Remove QSO targets
# mask = cat['DESI_TARGET'] & 2**2 ==0
# print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# cat = cat[mask]

# Require a minimum depth
min_depth = 800.
mask = cat['EFFTIME_LRG']>min_depth
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]

# # Julien's bad fibers list + my list of worst-performing fibers; bad_fibers-everest.ipynb
# # bad_fibers = np.loadtxt('/global/cfs/cdirs/desi/users/rongpu/spectro/everest/misc/bad_fibers_20211117.txt', dtype=int)
# bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211117.txt', dtype=int)
# print(len(bad_fibers))
# mask_bad = np.in1d(cat['FIBER'], bad_fibers)
# print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
# cat = cat[~mask_bad]
# print(len(cat), len(np.unique(cat['TARGETID'])))

FIBERSTATUS 338266 7165 0.020742203218587794
No data 338265 1 2.9562533627382e-06
Min depth 325438 12827 0.9620800260151064


In [12]:
# Redshift quality cut
cat['q'] = cat['ZWARN']==0
cat['q'] &= cat['Z']<1.5
cat['q'] &= cat['DELTACHI2']>15

print(np.sum(~cat['q'])/len(cat['q']))
# cat = cat[cat['q']]
# print(len(cat))

0.014841536636778742


In [13]:
mask_star = (cat['SPECTYPE']=='STAR') | (cat['Z']<0.0003)
cat['SPECTYPE'][mask_star] = 'STAR'
print(np.sum(mask_star)/len(mask_star))

0.015569785950011983


In [14]:
t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'], return_counts=True)
t['frac (%)'] = t['count']/len(cat)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

SPECTYPE,count,frac (%)
str6,int64,float64
STAR,5067,1.6
QSO,6891,2.1
GALAXY,313480,96.3


In [15]:
# Fraction of QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask)/len(mask))

0.01607372218364174


In [16]:
# Exclude QSO targets
mask = cat['DESI_TARGET'] & 2**2==0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

320207 0.9839262778163582


SPECTYPE,count,frac (%)
str6,int64,float64
QSO,3796,1.2
STAR,4793,1.5
GALAXY,311618,97.3


In [17]:
# Select QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

5231 0.01607372218364174


SPECTYPE,count,frac (%)
str6,int64,float64
STAR,274,5.2
GALAXY,1862,35.6
QSO,3095,59.2


In [18]:
# Downweight QSO targets and recalculate the fractions
mask_qso = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask_qso)/len(mask_qso))
cat['weight'] = 1.
cat['weight'][mask_qso] = 0.3

mask = cat['SPECTYPE']=='STAR'
print('Star fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))
mask = cat['SPECTYPE']=='QSO'
print('QSO fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))

0.01607372218364174
Star fraction: 1.52%
QSO fraction: 1.47%
